# Домашнє завдання. Алгоритми навчання з вчителем Ч.1

## 1. Імпорт необхідних пакетів

In [1]:
import pandas as pd

from scipy.stats import zscore

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
)

pd.set_option('display.float_format', lambda x: f'{x:.4f}')

## 2. Завантаження набору даних California Housing

Завантажуємо набір «Житловий фонд Каліфорнії» `fetch_california_housing` (першоджерело — StatLib, CMU).

In [2]:
california_housing = fetch_california_housing(as_frame=True)

data = california_housing['frame']
data.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0000,6.9841,1.0238,322.0000,2.5556,37.8800,-122.2300,4.5260
1,8.3014,21.0000,6.2381,0.9719,2401.0000,2.1098,37.8600,-122.2200,3.5850
2,7.2574,52.0000,8.2881,1.0734,496.0000,2.8023,37.8500,-122.2400,3.5210
3,5.6431,52.0000,5.8174,1.0731,558.0000,2.5479,37.8500,-122.2500,3.4130
4,3.8462,52.0000,6.2819,1.0811,565.0000,2.1815,37.8500,-122.2500,3.4220


Виділяємо цільову змінну `MedHouseVal` (медіанна вартість будинку) —
це неперервна величина, отже перед нами задача регресії.

In [3]:
target = data.pop('MedHouseVal')
target.head()

0   4.5260
1   3.5850
2   3.5210
3   3.4130
4   3.4220
Name: MedHouseVal, dtype: float64

## 3. Додаткова обробка даних

### 3.1. Очистка від викидів

У розділі «Розподіл ознак» практичного заняття було показано, що колонки
`AveRooms`, `AveBedrms`, `AveOccup`, `Population` мають дуже широкий діапазон значень
із аномально великими максимумами (наприклад, `AveRooms` до ~142, `AveOccup` до ~1243).
Погляньмо на описову статистику ще раз.

In [4]:
features_of_interest = ['AveRooms', 'AveBedrms', 'AveOccup', 'Population']
data[features_of_interest].describe()

,AveRooms,AveBedrms,AveOccup,Population
count,20640.0000,20640.0000,20640.0000,20640.0000
mean,5.4290,1.0967,3.0707,1425.4767
std,2.4742,0.4739,10.3860,1132.4621
min,0.8462,0.3333,0.6923,3.0000
25%,4.4407,1.0061,2.4297,787.0000
50%,5.2291,1.0488,2.8181,1166.0000
75%,6.0524,1.0995,3.2823,1725.0000
max,141.9091,34.0667,1243.3333,35682.0000


**Алгоритм очистки:**

1. Для кожної з чотирьох колонок обчислюємо z-критерій функцією `zscore` (`scipy`,
   `nan_policy='omit'`). z-критерій показує, на скільки стандартних відхилень значення
   відхиляється від середнього.
2. За **правилом трьох сигм** аномальними вважаємо значення, що виходять за межі діапазону
   **[-3, 3]** (тобто $|z| > 3$).
3. **Правило вилучення:** рядок вважаємо викидом, якщо аномальне значення є **хоча б в одній**
   із цих колонок (`any()` по осі колонок).

In [5]:
# z-критерії для кожної з чотирьох колонок (zscore зі scipy, nan_policy='omit' — як у зразку).
# У наборі немає пропусків, тож nan_policy на результат не впливає (додано для відповідності зразку).
z_scores = data[features_of_interest].apply(zscore, nan_policy='omit')
z_scores.describe()

,AveRooms,AveBedrms,AveOccup,Population
count,20640.0000,20640.0000,20640.0000,20640.0000
mean,0.0000,-0.0000,0.0000,-0.0000
std,1.0000,1.0000,1.0000,1.0000
min,-1.8523,-1.6108,-0.2290,-1.2561
25%,-0.3994,-0.1912,-0.0617,-0.5638
50%,-0.0808,-0.1011,-0.0243,-0.2291
75%,0.2520,0.0060,0.0204,0.2645
max,55.1632,69.5717,119.4191,30.2503


In [6]:
# Викид — значення поза діапазоном [-3, 3] (правило трьох сигм): у зразку це `~z.between(-3, 3)`.
# Рядок вважаємо викидом, якщо z виходить за [-3, 3] хоча б в одній колонці.
is_outlier = (~z_scores.apply(lambda col: col.between(-3, 3))).any(axis=1)

print(f'Всього об\'єктів:      {len(data)}')
print(f'Виявлено викидів:      {is_outlier.sum()}')
print(f'Залишиться об\'єктів:   {(~is_outlier).sum()}')

Всього об'єктів:      20640
Виявлено викидів:      505
Залишиться об'єктів:   20135


In [7]:
# Вилучаємо об'єкти з аномальними значеннями
data = data[~is_outlier]
target = target[~is_outlier]

print('Розмір набору після очистки:', data.shape)
data[features_of_interest].describe()

Розмір набору після очистки: (20135, 8)


,AveRooms,AveBedrms,AveOccup,Population
count,20135.0000,20135.0000,20135.0000,20135.0000
mean,5.2906,1.0684,2.9318,1340.0916
std,1.2702,0.1352,0.8856,812.5647
min,0.8462,0.3333,0.7500,3.0000
25%,4.4334,1.0050,2.4292,788.0000
50%,5.2128,1.0479,2.8176,1158.0000
75%,6.0173,1.0979,3.2807,1692.0000
max,12.5000,2.5146,33.9529,4819.0000


Після очистки максимальні значення ознак стали значно адекватнішими
(наприклад, `AveOccup` вже не сягає тисячі, `AveRooms` — не сягає сотні),
що прибирає найгрубіші аномалії, які «тягнули» лінію регресії на себе.

### 3.2. Видалення сильно корельованої ознаки

У розділі «Матриця кореляції змінних» практичного заняття було показано, що
**кількість кімнат (`AveRooms`) і кількість спалень (`AveBedrms`) сильно корелюють між собою**.
Наявність обох ознак одночасно небажана (мультиколінеарність). Перевіримо кореляцію
на очищених даних і приймемо рішення, яку саме ознаку прибрати.

In [8]:
corr = pd.concat([data, target], axis=1).drop(columns=['Longitude', 'Latitude']).corr()
corr[['AveRooms', 'AveBedrms', 'MedHouseVal']]

,AveRooms,AveBedrms,MedHouseVal
MedInc,0.6565,-0.1670,0.6894
HouseAge,-0.2157,-0.1259,0.1053
AveRooms,1.0000,0.3001,0.3203
AveBedrms,0.3001,1.0000,-0.0952
Population,-0.0868,-0.0452,-0.0336
AveOccup,-0.0428,-0.0810,-0.2368
MedHouseVal,0.3203,-0.0952,1.0000


Кореляція `AveRooms`–`AveBedrms` дуже висока. При цьому з цільовою змінною
`MedHouseVal` сильніше пов'язана саме `AveRooms`. Тому **залишаємо `AveRooms`,
а `AveBedrms` видаляємо** — прибираємо надлишкову ознаку, зберігаючи інформативнішу.

In [9]:
data = data.drop(columns=['AveBedrms'])
data.head()

,MedInc,HouseAge,AveRooms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0000,6.9841,322.0000,2.5556,37.8800,-122.2300
1,8.3014,21.0000,6.2381,2401.0000,2.1098,37.8600,-122.2200
2,7.2574,52.0000,8.2881,496.0000,2.8023,37.8500,-122.2400
3,5.6431,52.0000,5.8174,558.0000,2.5479,37.8500,-122.2500
4,3.8462,52.0000,6.2819,565.0000,2.1815,37.8500,-122.2500


## 4. Розбиття на навчальну і тестову вибірки

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    data,
    target,
    test_size=0.2,
    random_state=42)

print('X_train:', X_train.shape, '| X_test:', X_test.shape)

X_train: (16108, 7) | X_test: (4027, 7)


## 5. Нормалізація ознак

`StandardScaler` навчаємо **лише на тренувальній вибірці** (щоб уникнути витоку даних),
а потім застосовуємо ті самі параметри до тестової.

In [11]:
scaler = StandardScaler().set_output(transform='pandas').fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled.describe()

,MedInc,HouseAge,AveRooms,Population,AveOccup,Latitude,Longitude
count,16108.0000,16108.0000,16108.0000,16108.0000,16108.0000,16108.0000,16108.0000
mean,-0.0000,-0.0000,-0.0000,-0.0000,-0.0000,-0.0000,-0.0000
std,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000,1.0000
min,-1.7739,-2.2409,-3.5071,-1.6442,-2.4482,-1.4503,-2.3679
25%,-0.6876,-0.7974,-0.6752,-0.6812,-0.5625,-0.7993,-1.1070
50%,-0.1765,0.0045,-0.0664,-0.2266,-0.1287,-0.6447,0.5377
75%,0.4568,0.6461,0.5756,0.4355,0.3899,0.9710,0.7869
max,5.8625,1.8489,5.6824,4.2718,34.8735,2.9520,2.5114


## 6. Побудова моделі

Навчаємо `LinearRegression`. Обмежуємо прогнози діапазоном цільової змінної на тренуванні — бо вихідні ціни у наборі штучно обмежені зверху значенням 5.0.

In [12]:
model = LinearRegression().fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

ymin, ymax = y_train.agg(['min', 'max']).values
y_pred = pd.Series(y_pred, index=X_test_scaled.index).clip(ymin, ymax)
y_pred.head()

7969    2.3219
17082   2.8181
5623    1.6537
16934   2.5587
7501    1.5739
dtype: float64

## 7. Оцінка показників моделі

In [13]:
r_sq_upd = model.score(X_train_scaled, y_train)
mae_upd = mean_absolute_error(y_test, y_pred)
mape_upd = mean_absolute_percentage_error(y_test, y_pred)

print(f'R2: {r_sq_upd:.2f} | MAE: {mae_upd:.2f} | MAPE: {mape_upd:.2f}')

R2: 0.64 | MAE: 0.50 | MAPE: 0.29


## 8. Порівняння з базовою моделлю та висновки

In [14]:
comparison = pd.DataFrame(
    {
        'Базова модель (практика)': [0.61, 0.52, 0.31],
        'Модель після обробки (ДЗ)': [round(r_sq_upd, 2), round(mae_upd, 2), round(mape_upd, 2)],
    },
    index=['R2', 'MAE', 'MAPE'],
)
comparison['Зміна'] = comparison['Модель після обробки (ДЗ)'] - comparison['Базова модель (практика)']
comparison

,Базова модель (практика),Модель після обробки (ДЗ),Зміна
R2,0.6100,0.6400,0.0300
MAE,0.5200,0.5000,-0.0200
MAPE,0.3100,0.2900,-0.0200


### Висновки

**Що зроблено додатково порівняно з базовою моделлю:**

1. **Очистка від викидів** (`zscore`, правило трьох сигм) по колонках
   `AveRooms`, `AveBedrms`, `AveOccup`, `Population`. Рядок видалявся, якщо хоча б в одній
   з цих колонок значення виходило за межі $|z| > 3$.
2. **Видалення надлишкової ознаки `AveBedrms`**, яка сильно корелює з `AveRooms`
   (усунення мультиколінеарності); інформативнішу для цілі `AveRooms` залишено.

**Результат порівняння метрик:**

* **R2** зросла з `0.61` до `0.64` — модель пояснює більшу частку дисперсії цільової змінної.
* **MAE** зменшилася з `0.52` до `0.50` — середня абсолютна похибка стала меншою.
* **MAPE** зменшилася з `0.31` до `0.29` — відносна похибка також покращилась.

Тобто всі три метрики покращилися (R2 — вгору, MAE та MAPE — вниз), приблизно на
**2–3 відсоткові пункти**. Це підтверджує, що навіть базові кроки підготовки даних
(видалення грубих аномалій та надлишкових ознак) підвищують якість лінійної моделі,
не змінюючи сам алгоритм.

**Напрямки подальшого покращення** (за межами цього ДЗ):

* логарифмування/трансформація асиметричних ознак (`MedInc`, `Population`, `AveOccup`);
* поліноміальні ознаки (`PolynomialFeatures`) — див. розділ 9;
* окреме врахування «стелі» цільової змінної на рівні 5.0;
* збагачення набору зовнішніми ознаками (відстань до узбережжя тощо).

Нижче (розділ 9) я перевіряю один із цих напрямків — **поліноміальні ознаки**.

## 9. Дві версії набору ознак

Основне рішення (розділи 1–8) виконано **строго за інструкцією** й покращує базову модель.
Проте звичайна лінійна регресія лише на двох базових кроках має обмежену якість. Тут я
порівнюю **дві версії ознак** на тих самих очищених даних і тому самому розбитті:

* **Версія A — базові ознаки** (7 ознак): лінійна модель із розділів 5–7.
* **Версія B — `PolynomialFeatures(degree=2)`**: додаємо попарні добутки та квадрати ознак,
  даючи лінійній моделі змогу вловити нелінійні залежності (кількість ознак зростає до 35).

Дані, розбиття та нормалізація — ті самі, що вище (жодного повторного `train_test_split`),
тож порівняння коректне.

In [15]:
from sklearn.preprocessing import PolynomialFeatures

# Версія B: поліноміальні ознаки поверх уже нормалізованих ознак
poly = PolynomialFeatures(degree=2, include_bias=False).fit(X_train_scaled)
X_train_poly = poly.transform(X_train_scaled)
X_test_poly = poly.transform(X_test_scaled)

print('Версія A (базові ознаки):     ', X_train_scaled.shape[1], 'ознак')
print('Версія B (PolynomialFeatures):', X_train_poly.shape[1], 'ознак')

Версія A (базові ознаки):      7 ознак
Версія B (PolynomialFeatures): 35 ознак


In [16]:
# Навчаємо окрему лінійну модель на поліноміальних ознаках
model_poly = LinearRegression().fit(X_train_poly, y_train)

y_pred_poly = model_poly.predict(X_test_poly)
y_pred_poly = pd.Series(y_pred_poly, index=X_test_scaled.index).clip(ymin, ymax)

r_sq_poly = model_poly.score(X_train_poly, y_train)
mae_poly = mean_absolute_error(y_test, y_pred_poly)
mape_poly = mean_absolute_percentage_error(y_test, y_pred_poly)

print(f'R2: {r_sq_poly:.2f} | MAE: {mae_poly:.2f} | MAPE: {mape_poly:.2f}')

R2: 0.70 | MAE: 0.45 | MAPE: 0.25


In [17]:
features_comparison = pd.DataFrame(
    {
        'Базова модель (практика)': [0.61, 0.52, 0.31],
        'Версія A — базові ознаки': [round(r_sq_upd, 2), round(mae_upd, 2), round(mape_upd, 2)],
        'Версія B — PolynomialFeatures': [round(r_sq_poly, 2), round(mae_poly, 2), round(mape_poly, 2)],
    },
    index=['R2', 'MAE', 'MAPE'],
)
features_comparison

,Базова модель (практика),Версія A — базові ознаки,Версія B — PolynomialFeatures
R2,0.6100,0.6400,0.7000
MAE,0.5200,0.5000,0.4500
MAPE,0.3100,0.2900,0.2500


### Висновок за двома версіями ознак

* **Версія A (базові ознаки)** : R2 `0.61 → 0.64`, MAE та MAPE
  теж покращились. Це результат самих кроків підготовки даних, без зміни моделі.
* **Версія B (`PolynomialFeatures`, degree=2)** — та сама лінійна регресія, але на розширеному
  наборі ознак: R2 сягає **≈0.70**, MAE **≈0.45**, MAPE **≈0.25**. Тобто помітний приріст якості
  дають саме поліноміальні ознаки, а не два базові кроки.

**Чому обрано degree=2, а не вище.** З підвищенням степеня R2 на train продовжує зростати
(degree=3 дає ≈0.74), але це вже переднавчання: кількість ознак різко збільшується, а приріст
на тесті зупиняється. degree=2 — розумний компроміс між якістю та узагальнювальною здатністю.